# Clase 3 — Ambientes virtuales y gestión reproducible de dependencias

> **Pregunta guía:** si Git comparte nuestros archivos, ¿cómo logramos que otra
> computadora ejecute el proyecto con la misma versión de Python y las mismas
> bibliotecas?

En la clase 2 aprendimos a registrar y compartir cambios con Git. Hoy añadiremos
la pieza que Git no guarda: el **ambiente virtual** donde se ejecuta Python.

Primero construiremos y administraremos un ambiente virtual ejecutando cada paso de manera
explícita. Después analizaremos qué responsabilidades quedaron separadas y adoptaremos un
gestor integrado para coordinarlas.

**Resultado observable:** podrás crear, activar y reconstruir un ambiente virtual;
comparar dos formas de generar archivos de requisitos; y distinguir declaración, resolución,
instalación local y ejecución reproducible en `pcd-entregas-2026`.


## Quiz 1

**Código de acceso:** `RamaCommit2026`

El quiz estará disponible únicamente durante la franja indicada en Canvas.

## Antes de comenzar

Actualiza el repositorio público del curso desde su raíz:

```bash
git status
git switch main
git pull
```

Si `git status` muestra cambios tuyos, no ejecutes `git pull` a ciegas: conserva esos
cambios y pide apoyo para decidir cómo integrarlos.

**Prerrequisitos:** abrir Git Bash en Windows o la terminal en macOS/Linux, reconocer la
raíz de un repositorio y usar el flujo Git aprendido en la clase 2. No necesitas conocer
previamente ambientes virtuales, PyPI, `pip`, archivos de requisitos ni gestores de proyecto.


## 1. Ambientes virtuales

En la clase 2 Git nos permitió compartir archivos. Sin embargo, clonar el mismo código no
garantiza que se ejecute igual: la otra computadora también necesita un intérprete de Python
compatible y las bibliotecas correctas. Si esas piezas faltan o tienen otras versiones, el
programa puede fallar aunque el archivo `.py` sea idéntico.

Para entender el problema, separemos cuatro piezas que al principio suelen parecer una sola:

| Pieza | Qué es | Ejemplo de hoy |
|---|---|---|
| **Intérprete** | El programa que lee y ejecuta Python. | Python 3.12 |
| **Biblioteca** | Conjunto de funciones o clases reutilizables que resuelven una tarea. Describe lo que el código nos permite hacer. | `requests` permite preparar y enviar solicitudes HTTP. |
| **Dependencia** | Biblioteca externa que nuestro proyecto necesita y cuya instalación debemos controlar. | El proyecto requiere `requests>=2.32,<3`. |
| **Proyecto** | Código y configuración que persiguen un resultado común. | `pcd-entregas-2026` |

La palabra **paquete** puede significar varias cosas en Python y por eso no la usaremos como
categoría principal de esta tabla. Cuando hablemos de instalación, **paquete instalable** o
**distribución** será el archivo publicado en un índice como PyPI que `pip` descarga e
instala. **Biblioteca** describe la funcionalidad que usamos desde el código. En el ejemplo
`requests`, la biblioteca y la distribución comparten nombre; no siempre ocurre así.

Imagina dos proyectos en la misma computadora. El proyecto de análisis funciona con la
versión 1 de una biblioteca, mientras una API nueva requiere la versión 2. Si ambos usan la
misma instalación global, sólo una versión puede ocupar ese lugar. Actualizar para la API
puede romper el análisis; conservar la anterior puede impedir que la API inicie.

| Síntoma | Lo que realmente puede estar ocurriendo |
|---|---|
| `ModuleNotFoundError` | La biblioteca no está instalada en el Python que ejecutó el programa. |
| “En mi computadora sí funciona” | Cada persona tiene paquetes o versiones diferentes. |
| Un proyecto falla después de actualizar otro | Los dos compartían la instalación global. |
| VS Code y la terminal dan resultados distintos | Están usando intérpretes diferentes. |


### 1.1 Qué es un ambiente virtual y qué problema resuelve

Un **ambiente virtual de Python** es un directorio asociado a un intérprete y con un espacio
propio para instalar bibliotecas. Permite que cada proyecto controle su instalación sin
modificar la de otros proyectos en la misma computadora.

- **Aísla:** instalar una biblioteca allí no la instala para todos los proyectos.
- **Puede activarse:** la terminal puede dirigir temporalmente `python` y `pip` hacia él.
- **Es reconstruible:** contiene una instalación generada, no el trabajo original. Si la
  carpeta se pierde, podemos crearla nuevamente desde la configuración del proyecto.
- **Tiene límites:** no captura el sistema operativo, drivers, bases de datos ni servicios.

Compara ahora las dos situaciones de la figura. A la izquierda, ambos proyectos instalan
bibliotecas en el mismo espacio global y compiten por una sola versión. A la derecha, cada
proyecto tiene su propio ambiente virtual y conserva la versión que necesita. La etiqueta
`.venv/` es el nombre que usaremos para esa carpeta; se explicará en la subsección siguiente.

![Comparación entre una instalación global con conflicto de versiones y dos proyectos con ambientes virtuales separados](../assets/modulo-01-fundamentos/clase-03/ambientes-virtuales.svg)

*Figura 1. Un ambiente virtual separa el intérprete y los paquetes instalados para un
proyecto. En esta figura sólo analizamos **aislamiento**; la reproducibilidad se retomará al
final de la clase. Elaboración propia, licencia MIT.*

> **Idea clave:** un ambiente virtual es una instalación local y aislada. No es una
> máquina virtual, un contenedor ni una copia completa del sistema operativo.


### 1.2 `venv` y `.venv/` no son lo mismo

Python incluye un módulo llamado `venv` cuya función es **crear ambientes virtuales**. Al
ejecutarlo debemos indicar el directorio donde guardará la instalación. En este curso
elegiremos el nombre `.venv/` para ese directorio.

| Término | Qué representa |
|---|---|
| `venv` | La herramienta incluida en Python que crea el ambiente virtual. |
| `.venv/` | El directorio que contendrá el ambiente virtual de este proyecto. |

El punto inicial forma parte del nombre `.venv`. En macOS y Linux suele ocultar la carpeta
en listados normales; además, editores y herramientas reconocen esta convención. Podríamos
elegir otro nombre, pero usar siempre `.venv/` evita inconsistencias en el curso.

Cuando usemos `venv` para crear `.venv/` aparecerá una estructura semejante a ésta:

```text
.venv/
├── Scripts/        # Git Bash/Windows: python.exe, pip y activate
├── bin/            # macOS/Linux: python, pip y activate
├── Lib/ o lib/     # paquetes instalados para este ambiente virtual
└── pyvenv.cfg      # referencia al Python usado para crearlo
```

Windows y macOS/Linux usan nombres internos distintos, pero cumplen la misma función. No
edites estos archivos a mano. Más adelante eliminaremos y reconstruiremos toda la carpeta.

Podemos observar qué intérprete está ejecutando esta celda:


In [ ]:
import sys

print("Ejecutable:", sys.executable)
print("Python:", sys.version.split()[0])


Si la ruta contiene `.venv`, el notebook usa el ambiente virtual del proyecto docente.
La ruta exacta cambia entre sistemas; lo importante es reconocer **qué intérprete** está
ejecutando el código.

### 1.3 Herramientas para construir nuestro primer ambiente virtual

En las siguientes secciones crearemos un ambiente virtual ejecutando nosotros cada operación.
Llamaremos **flujo manual** a esta secuencia porque una herramienta crea el ambiente virtual,
otra instala las bibliotecas y nosotros mantenemos el archivo que permite repetir la
instalación. Estas son las cuatro piezas que utilizaremos:

| Término | Función | Qué no hace por sí solo |
|---|---|---|
| **PyPI** | Catálogo público donde se publican paquetes de Python. | No instala ni aísla paquetes. |
| **`pip`** | Instala paquetes desde PyPI u otras fuentes. | No crea un ambiente virtual. |
| **`venv`** | Módulo incluido en Python para crear ambientes virtuales. | No decide ni registra las dependencias. |
| **`requirements.txt`** | Archivo de texto con los nombres y versiones que `pip` debe instalar. | No crea el ambiente virtual ni se actualiza por sí solo. |

La secuencia importa: `venv` crea el lugar aislado, la activación dirige la terminal hacia
él, `pip` obtiene e instala paquetes y un archivo de requisitos puede describir qué volver a
instalar. PyPI es el catálogo remoto del que normalmente se descargan los paquetes.

```text
crear .venv → activar → pip consulta PyPI → instalar → registrar requisitos
```

### 1.4 Otras formas de crear ambientes virtuales

`venv` no es la única herramienta posible. Conviene conocer las familias principales antes
de elegir una, porque no todas administran las mismas piezas:

| Método | Qué crea | Cuándo suele aparecer | Papel en esta clase |
|---|---|---|---|
| `venv` | Un ambiente virtual de Python usando un intérprete ya instalado. | Proyectos Python sencillos y documentación básica. | Lo usaremos primero para ejecutar manualmente cada paso. |
| `virtualenv` | Ambientes virtuales mediante un paquete externo con opciones adicionales. | Proyectos antiguos o flujos que necesitan compatibilidad específica. | Sólo reconocerlo; no lo instalaremos. |
| Conda o Mamba | Ambientes que pueden incluir Python, bibliotecas y dependencias no escritas en Python. | Ciencia de datos con paquetes binarios o ecosistemas Conda. | Sólo reconocer su alcance. |

También es posible indicar a un editor como VS Code qué ambiente virtual debe usar, pero el
editor **no reemplaza** la herramienta que lo crea. En las siguientes secciones utilizaremos
`venv` directamente para entender creación, activación, instalación y reconstrucción. Más
adelante estudiaremos por qué los proyectos colaborativos suelen coordinar estas operaciones
con herramientas de mayor alcance.


## 2. Preparar el repositorio para crear un ambiente virtual manual

En las secciones 2, 3 y 4 construiremos el flujo tradicional **de manera manual**. Copiaremos
un script de práctica, protegeremos `.venv/` con `.gitignore`, crearemos un único ambiente
virtual con `venv`, lo activaremos, instalaremos `requests` y finalmente lo reconstruiremos
desde un archivo de requisitos. Al terminar podremos señalar qué pasos requieren coordinación.

Entra a la raíz de tu repositorio privado y abre una rama:

```bash
cd pcd-entregas-2026
git status
git switch main
git pull
git switch -c chore/configura-ambiente
```

Copia la actividad. El ejemplo supone que el repositorio público está junto al privado;
ajusta únicamente esa ruta si los guardaste en lugares diferentes.

```bash
mkdir -p actividades
cp -R ../proyecto-ciencia-datos-2026-2/labs/starters/clase-03-uv actividades/clase-03-uv
ls actividades/clase-03-uv/src
```

Debe aparecer `verificar_ambiente.py`. Este script importa `requests` y prepara una URL
local sin acceder a internet.


Antes de crear ambientes virtuales, abre o crea `.gitignore` en la raíz. Conserva las
reglas existentes y agrega:

```gitignore
.venv/
__pycache__/
```

Usaremos siempre el mismo nombre, `.venv/`, porque representa el ambiente virtual
actual del proyecto. Durante la demostración se elimina sólo esa carpeta generada y
se vuelve a crear; en proyectos normales, `.venv/` es también la convención más común.

**Checkpoint:** `pwd` y `git status` deben confirmar que sigues en la raíz de
`pcd-entregas-2026` y en la rama `chore/configura-ambiente`.


## 3. Crear y activar manualmente `.venv/` con `venv`

“Manualmente” significa que nosotros ejecutaremos y verificaremos cada operación: crear la
carpeta, activar el ambiente virtual y comprobar qué intérprete quedó seleccionado. Así
observaremos el mecanismo antes de estudiar herramientas que coordinan varios pasos.

Desde la raíz del repositorio ejecuta:

```bash
python -m venv .venv
```

- `python` elige el intérprete disponible en la terminal.
- `-m venv` le pide ejecutar el módulo estándar `venv`.
- `.venv` es el directorio que contendrá este ambiente demostrativo.

El nombre de la carpeta y el texto del prompt son decisiones distintas. Si quieres que al
activar aparezca `(pcd)` en vez de `(.venv)`, usa **este comando como alternativa al
anterior**, no ambos:

```bash
python -m venv --prompt pcd .venv
```

`--prompt pcd` personaliza sólo el indicador visual de la terminal; la carpeta continúa
llamándose `.venv/` y el aislamiento no cambia.

Crear no es activar. **Activar** modifica temporalmente la terminal para que `python` y
los comandos instalados apunten al ambiente virtual.

| Terminal | Comando de activación |
|---|---|
| Git Bash en Windows | `source .venv/Scripts/activate` |
| Zsh/Bash en macOS o Linux | `source .venv/bin/activate` |

El prompt mostrará `(.venv)` o el texto personalizado, por ejemplo `(pcd)`. No dependas
sólo de ese indicador: en la siguiente subsección verificaremos las rutas reales.


### 3.1 Comprobar qué Python y qué `pip` están activos

Ejecuta:

```bash
python -c "import sys; print(sys.executable)"
pip --version
```

La primera ruta debe contener `.venv`. La segunda debe mostrar que `pip` también
vive dentro de ese ambiente.

Como ya activamos y verificamos el ambiente virtual, desde este punto usaremos directamente
`pip`. La activación coloca primero en el `PATH` los comandos `python` y `pip` de `.venv/`.

**Checkpoint:** si alguna ruta apunta fuera de `.venv`, detente; instalar en ese
momento podría modificar otro Python.


### 3.2 Instalar `requests` dentro del ambiente virtual

Primero observa la falla esperada:

```bash
python actividades/clase-03-uv/src/verificar_ambiente.py
```

Debe aparecer `ModuleNotFoundError: No module named 'requests'`: el ambiente está aislado
y aún no contiene esa dependencia.

Ahora instálala desde PyPI y vuelve a ejecutar:

```bash
pip install "requests>=2.32,<3"
python actividades/clase-03-uv/src/verificar_ambiente.py
```

La salida debe incluir Python 3.12, la versión instalada de `requests` y una URL local.
`pip install` cambió `.venv`, pero todavía no dejó registrada la decisión en un
archivo del proyecto.


## 4. Registrar y reconstruir el ambiente virtual con archivos de requisitos

Un archivo de requisitos contiene líneas que `pip install` entiende. Por ejemplo:

```text
requests==2.34.2
```

El operador `==` exige una versión exacta. El nombre `requirements.txt` es una
convención, no una regla de Python.

### 4.1 Capturar todos los paquetes instalados con `pip freeze`

Captura el contenido completo del ambiente virtual activo:

```bash
pip freeze > requirements-freeze.txt
cat requirements-freeze.txt
```

`>` redirige la salida hacia un archivo y lo reemplaza si ya existe. `pip freeze` enumera
**lo que está instalado** en formato de requisitos; no explica qué decidió instalar el
equipo ni por qué aparece cada línea.

Como verificamos `pip --version` en la sección 3.1, sabemos que este comando lee el `pip`
instalado dentro de `.venv/`.


### 4.2 Inferir dependencias desde el código con `pipreqs`

`pipreqs` es una herramienta de terceros que examina imports del código. Instálala ahora
dentro del ambiente virtual activo y verifica que el comando esté disponible:

```bash
pip install pipreqs
pipreqs --help
```

Después ejecútala sobre la carpeta `src` y guarda el resultado en otro archivo:

```bash
pipreqs actividades/clase-03-uv/src --force --savepath requirements-imports.txt
cat requirements-imports.txt
```

Si el proyecto guarda imports dentro de notebooks `.ipynb`, añade la opción
`--scan-notebooks` y apunta a la carpeta que los contiene:

```bash
pipreqs notebooks --scan-notebooks --force --savepath requirements-notebooks.txt
```

No ejecutes ese segundo comando hoy si tu repositorio no tiene una carpeta `notebooks/`; es
la variante que usarías cuando los imports relevantes vivan en cuadernos.

- `pipreqs` examina imports y puede consultar PyPI para relacionarlos con distribuciones.
- `--force` permite reemplazar el archivo de salida.
- `--savepath` elige un nombre para no sobrescribir la captura anterior.
- `--scan-notebooks` incluye imports encontrados dentro de archivos `.ipynb`.

Este paso necesita red; si PyPI no responde, conserva el error y usa la salida docente.
Revisa siempre el resultado: un import puede tener nombre distinto en PyPI y el análisis
puede omitir dependencias cargadas dinámicamente, usadas sólo como comandos o necesarias
fuera de los archivos escaneados. El propio proyecto busca mantenedores; por eso lo usamos
para comprender el contraste, no como fuente canónica del curso.


### 4.3 Comparar una captura del ambiente con los imports del proyecto

| Método | Qué observa | Ventaja | Riesgo o límite |
|---|---|---|---|
| `pip freeze` | Todo lo instalado en el ambiente. | Captura exacta y fácil de reinstalar. | Incluye paquetes transitivos y herramientas que quizá no usa el programa. |
| `pipreqs` | Imports encontrados en el código. | Se aproxima a las dependencias directas. | Puede omitir o identificar mal dependencias; requiere revisión humana. |
| Declaración manual | Decisiones conscientes del proyecto. | Expresa intención y restricciones. | Hay que mantenerla al cambiar el código. |

En `requirements-freeze.txt` verás `requests` y las distribuciones que necesita para
funcionar. `pipreqs` no aparece allí porque instalamos esa herramienta **después** de tomar
la captura. En `requirements-imports.txt` debería aparecer principalmente `requests`, porque
es el import directo del script. Esa diferencia no significa que una lista esté “mal”:
responden preguntas distintas.


### 4.4 Eliminar y reconstruir `.venv/` desde la captura

Sal del ambiente actual y elimina **sólo** la carpeta generada `.venv/`:

```bash
deactivate
rm -rf .venv
```

Ejecuta ese `rm -rf` únicamente después de confirmar con `pwd` que estás en la raíz de
`pcd-entregas-2026`. `.venv/` contiene paquetes reconstruibles, no tu código ni tus
archivos Git.

Ahora crea de nuevo la misma carpeta y actívala según tu terminal:

```bash
python -m venv .venv
```

Si elegiste el prompt personalizado, repite en su lugar
`python -m venv --prompt pcd .venv`. En ambos casos reconstruyes la misma carpeta.

| Terminal | Comando |
|---|---|
| Git Bash en Windows | `source .venv/Scripts/activate` |
| Zsh/Bash en macOS o Linux | `source .venv/bin/activate` |

Después instala la captura y comprueba el script:

```bash
pip install -r requirements-freeze.txt
python actividades/clase-03-uv/src/verificar_ambiente.py
deactivate
```

`-r` significa “lee los requisitos desde este archivo”. Acabamos de reconstruir el mismo
ambiente virtual desde una lista, pero aún coordinamos manualmente creación, activación,
instalación y captura.

Ya demostramos que `.venv/` puede reconstruirse. Antes de cambiar de enfoque, confirma la
raíz y retira nuevamente **sólo** esa instalación generada:

```bash
pwd
rm -rf .venv
```

Los archivos de requisitos permanecen para la comparación. Más adelante, el gestor del
proyecto volverá a crear la misma ruta `.venv/`; no tendremos dos ambientes virtuales.


## 5. De herramientas separadas a un gestor de dependencias

El flujo manual funcionó y es importante saber reconocerlo. El problema no es que `venv`,
`pip` o los archivos de requisitos estén mal, sino que cada pieza tiene una responsabilidad
distinta y el equipo debe mantenerlas coordinadas:

| Pieza usada | Responsabilidad | Decisión que todavía recae en el equipo |
|---|---|---|
| `venv` | Crea el ambiente virtual. | Elegir el Python correcto, activarlo y reconstruirlo. |
| `pip` | Busca, instala, actualiza o elimina distribuciones. | Instalar dentro del ambiente correcto y registrar cada cambio. |
| Archivo de requisitos | Indica a `pip` qué instalar. | Mantenerlo alineado con el código y el ambiente virtual. |

En sentido estricto, `pip` es un **instalador de paquetes**. En conversaciones cotidianas
también se le llama manejador de paquetes porque instala, actualiza y elimina distribuciones.
Un **gestor de dependencias o de proyecto** agrega coordinación alrededor de esa instalación:
registra las dependencias directas, resuelve versiones compatibles, conserva el resultado y,
en algunos casos, también administra Python y el ambiente virtual.

### 5.1 Declarar, resolver y bloquear no significan lo mismo

- Una **dependencia directa** es una decisión del proyecto, como usar `requests`.
- Una **dependencia transitiva** es necesaria porque otra dependencia la utiliza. Nuestro
  código quizá no la importe directamente, pero debe instalarse.
- **Declarar** expresa qué paquetes y rangos de versiones acepta el proyecto.
- **Resolver** significa encontrar un conjunto de versiones que satisfaga simultáneamente
  las restricciones directas y transitivas.
- Un **lockfile** registra la resolución exacta elegida para poder repetirla. No es el ambiente
  virtual: es la descripción precisa que permite reconstruirlo.

Ahora podemos formular mejor lo que falta: necesitamos reducir la distancia entre declarar
una dependencia, resolver sus versiones, instalarla y compartir una receta reproducible.


## 6. Elegir un gestor integrado para el curso: `uv`

Existen varias rutas válidas. No instalaremos todas; la comparación sirve para ubicar qué
problema cubre cada familia:

| Enfoque | Ambiente virtual | Declaración y resolución | Archivo reproducible | Alcance habitual |
|---|---|---|---|---|
| `venv` + `pip` | Creación explícita | Coordinación manual | `requirements.txt` si se mantiene | Base disponible en Python |
| `pip-tools` + `venv` | Creación explícita | Compila requisitos declarados | `requirements*.txt` fijados | Equipos que desean permanecer cerca de `pip` |
| Pipenv | Gestionado por la herramienta | Integradas en el proyecto | `Pipfile.lock` | Aplicaciones Python |
| Poetry o PDM | Gestionado por la herramienta | Integradas en `pyproject.toml` | Lockfile propio | Aplicaciones y paquetes Python |
| Conda o Mamba | Gestionado por la herramienta | Python y paquetes, incluidos binarios | Archivo de ambiente | Ciencia de datos y ecosistemas Conda |
| **`uv`** | **`.venv/` gestionada por proyecto** | **Integradas en `pyproject.toml`** | **`uv.lock`** | **Ruta adoptada por el curso** |

Elegimos `uv` porque reúne las operaciones que acabamos de practicar y también puede localizar
o instalar la versión de Python que requiere el proyecto. No es indispensable para programar
en Python ni vuelve incorrecto el flujo anterior; es el acuerdo operativo del curso para que
todos trabajemos con una sola estructura.

| Operación que hicimos manualmente | Operación equivalente en el proyecto `uv` |
|---|---|
| Crear `.venv/` | `uv` la crea cuando el proyecto necesita sincronizarse. |
| Activar antes de ejecutar | `uv run ...` selecciona el ambiente virtual para ese comando. |
| `pip install paquete` | `uv add paquete` declara, resuelve e instala. |
| Mantener requisitos y versiones | `pyproject.toml` declara y `uv.lock` registra la resolución. |

Con este mapa ya tiene sentido instalar `uv`: no llega como un comando aislado, sino como una
forma de coordinar responsabilidades que ya observamos por separado.


## 7. Instalar `uv` y preparar Python 3.12

### Windows: PowerShell para instalar, Git Bash para trabajar

1. Abre **PowerShell** desde Inicio; no uses “Ejecutar como administrador”.
2. Ejecuta el instalador oficial:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

3. Cierra PowerShell, Git Bash y VS Code.
4. Abre una nueva Git Bash y verifica:

```bash
uv --version
```

Si el script está bloqueado pero WinGet está autorizado, usa en PowerShell
`winget install --id=astral-sh.uv -e`; después vuelve a Git Bash.

### macOS y Linux

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Cierra la terminal, abre una nueva y ejecuta el mismo comando de verificación.

Fuente: [instalación oficial de `uv`](https://docs.astral.sh/uv/getting-started/installation/),
consultada el 23 de agosto de 2026.

### 7.1 Localizar o instalar Python 3.12 con `uv`

`uv` distingue entre un Python **del sistema**, instalado previamente por el usuario o el
sistema operativo, y un Python **gestionado**, descargado por `uv`. Primero revisa qué
versiones ya están instaladas y busca una compatible con el curso:

```bash
uv python list --only-installed
uv python find 3.12
```

Si `uv python find 3.12` muestra una ruta, ya existe un intérprete compatible. Si indica que
no lo encontró, instálalo explícitamente y vuelve a buscarlo:

```bash
uv python install 3.12
uv python find 3.12
```

`uv` también puede descargar automáticamente una versión faltante cuando otro comando la
necesita. Durante la clase preferimos `uv python install 3.12` porque hace visible la descarga
y permite diagnosticar por separado problemas de red o permisos. Este comando instala Python;
todavía no crea `.venv/` ni declara dependencias del proyecto.

Fuente: [gestión oficial de versiones de Python con `uv`](https://docs.astral.sh/uv/guides/install-python/).


### 7.2 Errores comunes durante la instalación

| Síntoma | Diagnóstico seguro | Acción |
|---|---|---|
| Git Bash muestra `uv: command not found`. | La terminal no recibió el `PATH` nuevo. | Cierra todas las terminales y VS Code; vuelve a abrir Git Bash. |
| PowerShell bloquea el script. | Existe una política institucional. | Usa WinGet si está autorizado o pide apoyo; no cambies políticas globales. |
| El instalador no descarga. | Puede ser red, proxy, VPN o antivirus. | Conserva el mensaje exacto y usa la contingencia docente. |
| Funciona en PowerShell, no en Git Bash. | Cada shell leyó el `PATH` en otro momento. | Reinicia Git Bash; no edites `.bashrc` durante la clase. |
| `uv init` no reconoce `--bare`. | La instalación de `uv` es antigua respecto a la documentación de la clase. | Revisa `uv --version` y actualiza con el mismo método usado para instalar; no improvises otro flujo dentro del proyecto. |
| No aparece Python 3.12. | No existe una instalación compatible o no fue detectada. | Ejecuta `uv python find 3.12` y, si falla, `uv python install 3.12`. |
| VS Code usa otro Python. | El editor conserva un intérprete previo. | Después de crear `.venv/`, selecciona ese intérprete y reinicia el kernel. |

Para pedir ayuda comparte sistema operativo, terminal, comando y mensaje completo; nunca
compartas contraseñas, tokens ni rutas con datos personales.

Cuando `uv --version` funcione y `uv python find 3.12` devuelva una ruta, ya están listas
las dos herramientas necesarias. El siguiente paso será registrar en el repositorio qué
versión de Python y qué dependencias debe usar el proyecto.


## 8. Configurar `uv` en la raíz y entender sus archivos

No crearemos otro repositorio ni otra carpeta de proyecto. Inicializaremos `uv` en la raíz
del repositorio de entregas que ya usamos durante el semestre. Confirma primero que estás
allí y no dentro de `actividades/clase-03-uv/`:

```bash
pwd
git status
uv init --bare --python 3.12
uv python pin 3.12
```

- `--bare` crea sólo el `pyproject.toml` mínimo; no crea otro proyecto alrededor del repo.
- `--python 3.12` declara compatibilidad con Python 3.12.
- `uv python pin 3.12` crea `.python-version`.

No necesitamos `--name`: es opcional y, si se omite, `uv init` usa el nombre del directorio
actual, `pcd-entregas-2026`. Sólo tendría sentido agregarlo si el nombre técnico del proyecto
debiera ser diferente al de la carpeta.

Si ya existe `pyproject.toml`, no repitas `uv init`: ábrelo y confirma de dónde proviene.
La actividad queda bajo `actividades/`, pero la configuración y `.venv/` viven en la raíz
porque todo `pcd-entregas-2026` es un solo proyecto semestral.

### 8.1 Anatomía de los archivos y la carpeta del proyecto

`uv` coordina cuatro artefactos con responsabilidades diferentes. Que estén juntos no
significa que contengan lo mismo:

| Artefacto | Qué responde | Quién lo mantiene | ¿Se versiona? |
|---|---|---|:---:|
| `.python-version` | “¿Qué versión de Python debe preferir `uv` en este proyecto?” | `uv python pin`; una sola línea legible | Sí |
| `pyproject.toml` | “¿Qué es el proyecto y qué dependencias acepta?” | El equipo y comandos como `uv add` | Sí |
| `uv.lock` | “¿Qué versiones exactas resolvió `uv`, incluidas las transitivas?” | `uv` automáticamente; no se edita a mano | Sí |
| `.venv/` | “¿Qué intérprete y paquetes están instalados aquí, ahora?” | `uv sync` / `uv run` | **No** |

#### `.python-version`: la preferencia local de intérprete

Después de `uv python pin 3.12`, el archivo contiene simplemente:

```text
3.12
```

No contiene Python ni instala nada. Le indica a `uv` qué versión debe buscar o descargar al
crear el ambiente virtual. En cambio, `requires-python = ">=3.12"` dentro de
`pyproject.toml` declara qué versiones son compatibles con el proyecto. Preferencia local y
compatibilidad son decisiones relacionadas, pero no idénticas.

#### `pyproject.toml`: identidad y dependencias declaradas

**TOML** es un formato de texto para configuración. No es código Python y no se ejecuta.
Sus reglas básicas bastan para leer este archivo:

- `[project]` abre una tabla o sección;
- `name = "pcd-entregas-2026"` asigna un valor a una clave;
- los textos van entre comillas y las listas entre corchetes;
- `#` inicia un comentario.

El archivo mínimo creado por `uv init --bare` se parece a éste:

```toml
[project]
name = "pcd-entregas-2026"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = []
```

`dependencies = []` significa que todavía no declaramos paquetes. En la sección 9, `uv add`
modificará esa lista por nosotros. Este archivo expresa la **intención compatible** del
proyecto; no pretende enumerar cada paquete transitivo con una versión exacta.

#### `uv.lock`: resolución exacta y reproducible

El lockfile aparece cuando `uv` resuelve el proyecto mediante `uv add`, `uv lock`, `uv sync`
o `uv run`. Registra las versiones exactas de dependencias directas y transitivas y las
condiciones necesarias para distintos sistemas. Es un archivo TOML legible, pero su formato
pertenece a `uv`: se versiona y **no se edita manualmente**.

La relación completa será:

```text
.python-version ─┐
pyproject.toml  ──┼─→ uv resuelve → uv.lock → uv sincroniza → .venv/
código del repo ──┘                                      ↓
                                                   uv run python
```

`pyproject.toml` y `uv.lock` permiten acordar el proyecto; `.venv/` es sólo la instalación
local que `uv` puede reconstruir a partir de ese acuerdo.


## 9. Declarar, resolver e instalar `requests` con `uv add`

Ejecuta primero el script sin declarar `requests`:

```bash
uv run python actividades/clase-03-uv/src/verificar_ambiente.py
```

La falla por `requests` vuelve a ser esperada. Ahora registra la decisión:

```bash
uv add "requests>=2.32,<3"
uv run python actividades/clase-03-uv/src/verificar_ambiente.py
```

`uv add "requests>=2.32,<3"` se lee como “agrega `requests` a este proyecto y acepta
versiones desde 2.32 hasta antes de 3”. Realiza tres acciones coordinadas:

1. agrega `requests` a `pyproject.toml` como dependencia directa;
2. resuelve versiones y actualiza `uv.lock`;
3. crea o sincroniza `.venv/` con esa resolución.

Comprueba con:

```bash
git status --short
uv tree --depth 1
cat pyproject.toml
```

Ahora `dependencies` ya contiene la decisión directa sobre `requests`. `uv.lock` además
incluye las versiones exactas que `requests` necesita de forma transitiva. En los proyectos
del curso, **agrega bibliotecas con `uv add`**. Instalar algo directamente dentro de
`.venv/` no registraría la decisión y dejaría el ambiente desalineado con el proyecto.


### 9.1 ¿Por qué `uv run python`?

El comando se lee de izquierda a derecha:

```text
uv run       python          ruta/al/script.py
└─ prepara   └─ intérprete   └─ argumento que recibe Python
   proyecto    de .venv
```

Cuando se ejecuta dentro de un proyecto, `uv run`:

1. localiza el `pyproject.toml` más cercano;
2. comprueba que `uv.lock` y la declaración estén sincronizados;
3. crea o actualiza `.venv/` si hace falta;
4. busca `python` dentro de ese ambiente virtual;
5. ejecuta el script sin modificar permanentemente tu terminal.

Por eso no necesitamos `source .venv/.../activate`. Si escribieras solamente `python`,
la terminal podría usar el Python global. Con `uv run python` hacemos explícito que se
debe usar el ambiente administrado por el proyecto.

Para exigir que el lockfile no cambie durante la ejecución usa:

```bash
uv run --locked python actividades/clase-03-uv/src/verificar_ambiente.py
```

Fuente: [ejecución de comandos en proyectos `uv`](https://docs.astral.sh/uv/concepts/projects/run/).


## 10. Verificar que `uv` puede reconstruir el ambiente virtual

Comprueba el proyecto recomendado:

```bash
uv lock --check
uv sync --locked
uv run --locked python actividades/clase-03-uv/src/verificar_ambiente.py
```

- `uv lock --check` confirma que el lockfile corresponde a la declaración.
- `uv sync --locked` sincroniza `.venv/` sin permitir cambios al lockfile.
- `uv run --locked` ejecuta dentro del estado acordado.

> **Nota importante sobre `requirements*.txt`:** estos archivos no son incorrectos ni han
> desaparecido del ecosistema Python. Muchas plataformas y proyectos todavía los usan. En
> **`pcd-entregas-2026`**, sin embargo, fueron artefactos didácticos para comprender el flujo
> manual. La fuente canónica será `pyproject.toml` junto con `uv.lock`. Conservar ambos
> enfoques como fuentes editables permitiría que se contradijeran.

Retira los archivos demostrativos antes de preparar el commit:

```bash
rm -f requirements-freeze.txt requirements-imports.txt requirements-notebooks.txt
git status --short
```

El ambiente manual ya se eliminó para demostrar que era reconstruible. Después de
`uv sync --locked`, la misma carpeta `.venv/` vuelve a existir, ahora gestionada por `uv`.


## 11. Versionar los archivos reproducibles y cerrar la rama

| Sí se versiona | No se versiona |
|---|---|
| `.gitignore` | `.venv/` |
| `.python-version` | `__pycache__/` y `*.pyc` |
| `pyproject.toml` | Los `requirements-*.txt` demostrativos |
| `uv.lock` | Paquetes copiados a mano |
| `actividades/clase-03-uv/README.md` y `src/verificar_ambiente.py` | Credenciales o archivos personales |

La rama `chore/configura-ambiente` **ya se creó en la sección 2**. No abras otra. Comprueba que
sigues en ella y prepara únicamente las rutas esperadas:

```bash
git branch --show-current
git add .gitignore .python-version pyproject.toml uv.lock actividades/clase-03-uv
git diff --cached --name-status
git commit -m "chore: configura ambiente reproducible con uv"
git push -u origin chore/configura-ambiente
```

Abre el PR, verifica que no incluya ningún ambiente virtual e intégralo. Después vuelve a
`main` y actualízala:

```bash
git switch main
git pull
git status
```

El comando `git branch -d chore/configura-ambiente` es una **limpieza local opcional después del
merge**: elimina la rama ya integrada; no crea una rama nueva. `-d` se niega a borrarla si
Git no la reconoce como integrada. Puedes ejecutarlo al final, pero no es requisito para
entender `uv`.

Esta práctica ejercita Git, pero no es una entrega en Canvas.


## 12. Diagnosticar errores frecuentes

| Síntoma | Causa probable | Comprobación/acción |
|---|---|---|
| La activación dice que el archivo no existe. | Usaste la ruta de otro sistema o estás fuera de la raíz. | Ejecuta `pwd` y usa la fila correspondiente a tu terminal. |
| `pip` instala fuera del ambiente manual. | No se activó o se abrió otra terminal. | Revisa `python -c "import sys; print(sys.executable)"` antes de instalar. |
| `pipreqs` no encuentra o agrega un paquete incorrecto. | El import no coincide con el nombre de distribución o está fuera de la ruta escaneada. | Revisa el código y el archivo generado; no aceptes automáticamente la salida. |
| `No pyproject.toml found`. | Ejecutaste `uv` fuera de la raíz del proyecto. | Usa `pwd`, `ls` y `git status`; vuelve a `pcd-entregas-2026`. |
| `ModuleNotFoundError` después de `uv add`. | Usaste otro Python o proyecto. | Ejecuta con `uv run python ...` desde la raíz y revisa `uv tree`. |
| `.venv/` aparece en `git status`. | Falta la regla raíz de `.gitignore`. | Agrega la regla y prepara staging con rutas explícitas, no con `git add .`. |
| `requests` funciona hasta que se reconstruye `.venv/`. | Se instaló sin declararlo en el proyecto. | Ejecuta `uv add requests` y verifica el cambio en `pyproject.toml`. |

Conserva el error exacto, confirma ruta e intérprete y cambia una sola cosa a la vez.


## Recopilación: cómo se conecta todo

La clase avanzó por cuatro etapas. Cada una respondió una pregunta que la anterior dejó
abierta:

| Etapa | Pregunta | Respuesta construida en clase |
|---|---|---|
| 1. Aislamiento | ¿Cómo evitamos que dos proyectos mezclen bibliotecas? | Cada proyecto utiliza su propio ambiente virtual `.venv/`. |
| 2. Flujo manual | ¿Qué ocurre dentro de ese ambiente virtual? | `venv` lo crea, la activación selecciona sus comandos y `pip` instala las dependencias. |
| 3. Coordinación | ¿Cómo compartimos algo que puede reconstruirse? | Separamos dependencias directas, resolución exacta e instalación local. |
| 4. Flujo del curso | ¿Cómo mantenemos coordinadas esas piezas? | `uv` gestiona Python y `.venv/`, actualiza los archivos del proyecto y ejecuta dentro del ambiente virtual. |

### Qué representa cada artefacto al terminar

| Artefacto | Responsabilidad | ¿Viaja con Git? |
|---|---|:---:|
| Código `.py` y `.ipynb` | Instrucciones y análisis del proyecto. | Sí |
| `.python-version` | Versión de Python preferida para este repositorio. | Sí |
| `pyproject.toml` | Identidad, compatibilidad y dependencias directas aceptadas. | Sí |
| `uv.lock` | Versiones exactas resueltas, incluidas dependencias transitivas. | Sí |
| `.venv/` | Python y bibliotecas instalados físicamente en una computadora. | **No** |

### Los dos flujos completos

```text
Manual:
crear .venv → activar → pip install → pip freeze → pip install -r → python script.py

Proyecto del curso:
localizar Python → uv init → uv python pin → uv add → uv sync --locked → uv run python
```

En otra computadora no copiamos `.venv/`. La persona clona los archivos versionados, ejecuta
`uv sync --locked` para reconstruir el ambiente virtual y usa `uv run python` para ejecutar
con ese estado. En la clase 4 reutilizaremos exactamente este proyecto para instalar y
ejecutar FastAPI; no crearemos otro ambiente virtual.

![Flujo de reproducibilidad desde los archivos versionados hasta la ejecución dentro del ambiente virtual](../assets/modulo-01-fundamentos/clase-03/reproducibilidad-uv.svg)

*Figura 2. Git comparte `.python-version`, `pyproject.toml` y `uv.lock`; `uv sync --locked`
reconstruye `.venv/` localmente y `uv run python` ejecuta dentro de ese ambiente virtual.
Elaboración propia, licencia MIT.*


## Lectura extra recomendada — linters y formatters

> Esta sección es opcional y está separada del flujo de ambientes virtuales. No instalaremos
> estas herramientas en la clase 3.

Un **linter** analiza el código sin ejecutarlo para señalar patrones sospechosos, errores
comunes o convenciones incumplidas. Un **formatter** reescribe su presentación para aplicar
un estilo consistente. Ninguno sustituye pruebas, nombres claros ni revisión humana.

| Recurso | Tipo | Para explorar después |
|---|---|---|
| [Ruff](https://docs.astral.sh/ruff/) | Linter y formatter | Reglas rápidas y una sola herramienta para ambas tareas. |
| [Black](https://black.readthedocs.io/en/stable/) | Formatter | Un estilo uniforme con pocas decisiones manuales. |
| [PEP 8](https://peps.python.org/pep-0008/) | Guía de estilo | Convenciones de legibilidad en código Python. |

La pregunta importante no es “¿qué herramienta gana?”, sino qué problema resuelve cada una
y en qué momento una revisión automática aporta valor al equipo.


## 📚 Referencias

- [Python: creación de ambientes virtuales con `venv`](https://docs.python.org/3/library/venv.html)
- [`pip freeze`: referencia oficial](https://pip.pypa.io/en/stable/cli/pip_freeze/)
- [`pip`: archivos de requisitos](https://pip.pypa.io/en/stable/user_guide/#requirements-files)
- [`pipreqs`: repositorio y opciones](https://github.com/bndr/pipreqs)
- [`uv`: instalación oficial](https://docs.astral.sh/uv/getting-started/installation/)
- [`uv`: instalar y gestionar versiones de Python](https://docs.astral.sh/uv/guides/install-python/)
- [`uv`: migración desde `pip`](https://docs.astral.sh/uv/guides/migration/pip-to-project/)
- [`uv`: estructura y archivos de un proyecto](https://docs.astral.sh/uv/concepts/projects/layout/)
- [`uv`: trabajar con proyectos](https://docs.astral.sh/uv/guides/projects/)
- [`uv`: ejecución dentro del proyecto](https://docs.astral.sh/uv/concepts/projects/run/)
- [`uv`: referencia de comandos](https://docs.astral.sh/uv/reference/cli/)

Los procedimientos se contrastaron con la documentación disponible el 23 de agosto de
2026. `pipreqs` se presenta como herramienta de contraste y no como fuente canónica.
